# Lendo o arquivo e ajustando os tipos colunares 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# adaptando os tippos de dados para a manipulação
data = pd.read_csv(
    "dataset_avc.csv",
    dtype={
        "id": "int",
        "gender": "category",
        "ever_married": "category",
        "work_type": "category",
        "Residence_type": "category",
        "smoking_status": "category",
        "hypertension": "category",
        "heart_disease": "category",
        "stroke": "category",
    },
)



In [ ]:
# ---------------------------------------------------------------
# Medidas de centralizacao: media, mediana e moda
# (age, avg_glucose_level, bmi)
# ---------------------------------------------------------------
variaveis = ["age", "avg_glucose_level", "bmi"]

centralizacao = pd.DataFrame({
    "media":   data[variaveis].mean(),
    "mediana": data[variaveis].median(),
    "moda":    data[variaveis].mode().iloc[0],
})
print("== Centralizacao (base completa) ==")
centralizacao.round(2)


In [ ]:
# ---------------------------------------------------------------
# Medidas de posicao: Q1, Q2 (mediana) e Q3
# comparando pacientes COM e SEM AVC (coluna stroke)
# ---------------------------------------------------------------

# quartis na base completa
quartis_geral = data[variaveis].quantile([0.25, 0.50, 0.75]).T
quartis_geral.columns = ["Q1", "Q2", "Q3"]
print("== Quartis - base completa ==")
print(quartis_geral, "\n")

# quartis por grupo de AVC (0 = sem AVC, 1 = com AVC)
for valor, sub in data.groupby("stroke", observed=True):
    nome = "com_AVC" if str(valor) == "1" else "sem_AVC"
    q = sub[variaveis].quantile([0.25, 0.50, 0.75]).T
    q.columns = ["Q1", "Q2", "Q3"]
    print(f"== Quartis - {nome} (n = {len(sub)}) ==")
    print(q, "\n")

# tambem da' pra ver alguns percentis extras de uma vez:
percentis = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
print("== Percentis - base completa ==")
display(data[variaveis].quantile(percentis).T.round(2))


In [ ]:
# ---------------------------------------------------------------
# Medidas de dispersao: variancia, desvio padrao e amplitude
# (age, avg_glucose_level, bmi)
# ---------------------------------------------------------------

dispersao = pd.DataFrame({
    "variancia":    data[variaveis].var(),      # ddof=1 (amostral) por padrao
    "desvio_padrao": data[variaveis].std(),     # raiz da variancia
    "minimo":       data[variaveis].min(),
    "maximo":       data[variaveis].max(),
})
dispersao["amplitude"] = dispersao["maximo"] - dispersao["minimo"]
# coef. de variacao (%) -> compara dispersao entre variaveis de escalas diferentes
dispersao["cv_%"] = 100 * data[variaveis].std() / data[variaveis].mean()

print("== Dispersao (base completa) ==")
print(dispersao.round(2), "\n")

# por grupo de AVC
for valor, sub in data.groupby("stroke", observed=True):
    nome = "com_AVC" if str(valor) == "1" else "sem_AVC"
    d = pd.DataFrame({
        "variancia":     sub[variaveis].var(),
        "desvio_padrao": sub[variaveis].std(),
        "minimo":        sub[variaveis].min(),
        "maximo":        sub[variaveis].max(),
    })
    d["amplitude"] = d["maximo"] - d["minimo"]
    print(f"== Dispersao - {nome} (n = {len(sub)}) ==")
    print(d.round(2), "\n")


In [ ]:
# ---------------------------------------------------------------
# Pre-processamento: duplicados e valores nulos
# ---------------------------------------------------------------

# duplicados (linha inteira e por id, que deveria ser unico)
print("Linhas duplicadas:", data.duplicated().sum())
print("IDs duplicados:", data["id"].duplicated().sum())

# valores nulos por coluna
print("\nValores nulos por coluna:")
print(data.isnull().sum())

# bmi e a unica coluna com nulos (201 registros, ~4% da base)
mediana_bmi = data["bmi"].median()
qtd_nulos_bmi = data["bmi"].isnull().sum()
data["bmi"] = data["bmi"].fillna(mediana_bmi)
print(f"\nbmi: {qtd_nulos_bmi} nulos preenchidos com a mediana ({mediana_bmi:.2f})")

# smoking_status tem a categoria "Unknown", que representa dado ausente
print("\nDistribuicao de smoking_status:")
print(data["smoking_status"].value_counts())

In [ ]:
# ---------------------------------------------------------------
# Correlacao entre age, avg_glucose_level e bmi
# ---------------------------------------------------------------
import seaborn as sns
# correlacao de Pearson -> mede relacao LINEAR entre as variaveis
print("== Correlacao de Pearson ==")
print(data[variaveis].corr(method="pearson").round(2))

# correlacao de Spearman -> mede relacao monotonica (nao precisa ser linear)
print("\n== Correlacao de Spearman ==")
print(data[variaveis].corr(method="spearman").round(2))

plt.figure(figsize=(5, 4))
sns.heatmap(data[variaveis].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de correlacao (Pearson)")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------
# Padronizacao (z-score) de age, avg_glucose_level e bmi
# ---------------------------------------------------------------

# StandardScaler: transforma cada variavel para media 0 e desvio padrao 1
scaler = StandardScaler()
data_padronizado = pd.DataFrame(
    scaler.fit_transform(data[variaveis]),
    columns=[f"{c}_z" for c in variaveis],
)

# media e desvio ORIGINAIS, pra comparar com o resultado padronizado
print("== Antes da padronizacao (media e desvio originais) ==")
print(data[variaveis].agg(["mean", "std"]).round(2))

# apos a padronizacao, media deve ficar ~0 e desvio ~1 pras tres variaveis
print("\n== Depois da padronizacao (media ~0, desvio ~1) ==")
print(data_padronizado.agg(["mean", "std"]).round(2))

data_padronizado.round(2).head()

In [ ]:
import seaborn as sns

# histogramas das variaveis numericas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, variaveis):
    ax.hist(data[col], bins=30, color="steelblue", edgecolor="white")
    ax.set_title(col)
plt.suptitle("Distribuicao das variaveis numericas")
plt.tight_layout()
plt.show()

# boxplots por grupo de AVC (0 = sem AVC, 1 = com AVC)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, variaveis):
    data.boxplot(column=col, by="stroke", ax=ax)
    ax.set_title(col)
plt.suptitle("Distribuicao por grupo de AVC")
plt.tight_layout()
plt.show()



# pergunta 1

In [ ]:

# Configuração visual (estilo limpo conforme a imagem de referência)
sns.set_theme(style="ticks")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Configuração de estilo
sns.set_theme(style="ticks")
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# 1. Tratamento e Limpeza
cols_p1 = ["age", "stroke", "hypertension"]
df_p1 = data[cols_p1].dropna().copy()

for col in cols_p1:
    df_p1[col] = pd.to_numeric(df_p1[col], errors="coerce")
df_p1 = df_p1.dropna()

# 2. Agrupamento: Calcula a Taxa Real de AVC (%) para cada idade e status de hipertensão
df_taxas = (
    df_p1.groupby(["age", "hypertension"])["stroke"]
    .agg(taxa_avc="mean", total="count")
    .reset_index()
)
df_taxas["taxa_avc_pct"] = df_taxas["taxa_avc"] * 100

# Considera apenas idades com volume mínimo de dados (ex: >= 5 pacientes) para evitar ruído
df_taxas = df_taxas[df_taxas["total"] >= 3]

# 3. Separação dos grupos para cálculo de Pearson
sem_hip = df_taxas[df_taxas["hypertension"] == 0]
com_hip = df_taxas[df_taxas["hypertension"] == 1]

r_sem = sem_hip["age"].corr(sem_hip["taxa_avc_pct"], method="pearson")
r_com = com_hip["age"].corr(com_hip["taxa_avc_pct"], method="pearson")

# 4. Plota o Scatter Plot com Reta de Pearson
plt.figure(figsize=(8, 5))

# Grupo: Sem Hipertensão
sns.regplot(
    data=sem_hip,
    x="age",
    y="taxa_avc_pct",
    label=f"Sem Hipertensão (Pearson: {r_sem:.2f})",
    color="seagreen",
    scatter_kws={"alpha": 0.6, "s": 35},
    line_kws={"linewidth": 2.5},
    ci=None,
)

# Grupo: Com Hipertensão
sns.regplot(
    data=com_hip,
    x="age",
    y="taxa_avc_pct",
    label=f"Com Hipertensão (Pearson: {r_com:.2f})",
    color="crimson",
    scatter_kws={"alpha": 0.6, "s": 35},
    line_kws={"linewidth": 2.5},
    ci=None,
)

plt.title(
    "Taxa de Incidência de AVC (%) por Idade e Hipertensão",
    fontsize=12,
    pad=15,
)
plt.xlabel("Idade (anos)")
plt.ylabel("Incidência de AVC (%)")
plt.legend(title="Condição Cardiovascular", loc="upper left")
plt.tight_layout()
plt.show()